In [1]:
from loguru import logger
from plant_care.config import get_env, load_config
from plant_care.data_processor import DataProcessor
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder.getOrCreate()

# Load config
env = get_env()
cfg = load_config("../project_config.yml", env)

processor = DataProcessor(spark, cfg)

In [ ]:
# Parse article HTML into tagged text blocks and save to a table
parsed_df = processor.parse_all_articles(save_table=True)
logger.info(f"Parsed articles: {parsed_df.count()}")
parsed_df.printSchema()

2026-11-17 14:46:04.404 | INFO     | plant_care.data_processor:parse_all_articles:120 - Parsed 118 articles
2026-11-17 14:46:12.789 | INFO     | plant_care.data_processor:parse_all_articles:124 - Saved parsed articles to llmops_dev.srinidhi_ilango98.plant_care_parsed
2026-11-17 14:46:13.587 | INFO     | __main__:<module>:3 - Parsed articles: 118


root
 |-- slug: string (nullable = false)
 |-- url: string (nullable = true)
 |-- blocks: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- tag: string (nullable = true)
 |    |    |-- text: string (nullable = true)



In [4]:
# Review the first blocks of one sample article
sample = spark.table(processor.parsed_table).first()
logger.info(f"Sample article: {sample['slug']} ({len(sample['blocks'])} blocks)")
for block in sample["blocks"][:10]:
    print(f"[{block['tag']}] {block['text'][:200]}")

2026-11-17 14:46:20.170 | INFO     | __main__:<module>:3 - Sample article: african-violet (23 blocks)


[h2] Overview
[p] The African Violet (Saintpaulia) is one of the most beloved indoor flowering plants in the world, prized for its ability to produce vibrant clusters of purple, pink, or white flowers year-round. Howev
[p] Native to the humid cloud forests of Tanzania and Kenya, African Violets grow in rocky crevices or on the forest floor, sheltered from direct sun and heavy downpours. They absorb moisture from the hum
[h3] Light
[p] Light is the absolute critical factor for getting an African Violet to bloom. They require bright, indirect light. An east-facing window is ideal because it provides gentle morning sun and bright ambi
[h3] Temperature
[p] These plants prefer the exact same temperatures that humans find comfortable: 65°F to 80°F (18°C - 27°C). They are highly sensitive to sudden temperature changes. Keep them away from cold, drafty wind
[h3] Humidity
[p] Originating from cloud forests, they thrive in high humidity (50% to 70%). While they will tolerate average household hu

In [8]:
t = spark.table("llmops_dev.srinidhi_ilango98.plant_care")
print("rows:", t.count(), "with volume_path:", t.filter("volume_path IS NOT NULL").count())

rows: 143 with volume_path: 118


In [6]:
row = spark.table("llmops_dev.srinidhi_ilango98.plant_care").filter("volume_path IS NOT NULL").select("slug", "volume_path").first()
print(row.volume_path)
print(os.path.exists(row.volume_path))
display(dbutils.fs.ls(row.volume_path.rsplit("/", 1)[0]))

/Volumes/llmops_dev/srinidhi_ilango98/plant_care/articles/african-violet.html
True


NotFound: Not Found